<a href="https://colab.research.google.com/github/ishan-bharath/MIA-AI-Therapist-/blob/main/MIA_GPT2_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece


In [ ]:
from google.colab import files
import pandas as pd

# Upload your CSV file
print("Click 'Choose Files' and select therapy_data.csv from your computer:")
uploaded = files.upload()

# Now the file is in Colab's /content/ folder
data_path = "therapy_data.csv"

print("✓ File uploaded successfully!")


In [ ]:
import pandas as pd

df = pd.read_csv(data_path)

print("Columns:", df.columns.tolist())
print("Total rows:", len(df))
print(df.head(3))

# Basic cleaning: drop rows with missing values
df = df.dropna(subset=["input", "output"])
print("Rows after dropping NaNs:", len(df))

# Build a single text field for GPT-2
def build_conversation(row):
    user = str(row["input"]).strip()
    mia = str(row["output"]).strip()
    # Format as conversation
    return f"User: {user}\nMIA: {mia}\n\n"

df["text"] = df.apply(build_conversation, axis=1)

# Quick sanity check
print("\nExample formatted conversation:\n")
print(df["text"].iloc[0])


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# Create HF Dataset from pandas
dataset = Dataset.from_pandas(df[["text"]])

# Split into train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("Train examples:", len(train_dataset))
print("Eval examples:", len(eval_dataset))

# Load GPT-2 tokenizer
model_name = "gpt2"  # small, good for Colab
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 has no pad token; we set it to eos token
tokenizer.pad_token = tokenizer.eos_token

max_length = 256  # enough for your short dialogues

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("✓ Tokenization complete!")
print("\nFirst tokenized example (first 10 tokens):")
print(tokenized_train[0]["input_ids"][:10])



In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
import os

model = AutoModelForCausalLM.from_pretrained(model_name)

# Align padding token ids
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.eos_token_id

# Create output directory for saving models
model_output_dir = "/content/mia_gpt2_model"
os.makedirs(model_output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=model_output_dir,
    num_train_epochs=3,          # 3 epochs for 30 examples
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",       # changed from evaluation_strategy
    save_strategy="epoch",
    logging_steps=5,
    learning_rate=5e-5,
    warmup_steps=10,
    weight_decay=0.01,
    fp16=True,                   # use GPU half precision
    report_to="none",            # disable wandb
)

print("✓ Model loaded and training args configured!")
print(f"Model output directory: {model_output_dir}")


In [ ]:
import torch

# Labels are the same as input_ids for language modeling
def to_lm_labels(batch):
    batch["labels"] = batch["input_ids"].copy()
    return batch

tokenized_train_lm = tokenized_train.map(to_lm_labels, batched=True)
tokenized_eval_lm = tokenized_eval.map(to_lm_labels, batched=True)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_lm,
    eval_dataset=tokenized_eval_lm,
)

print("Starting training…")
trainer.train()
print("\n✓ Training complete!")


In [ ]:
# Save final model and tokenizer
save_path = model_output_dir + "/final"
import os
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✓ Model and tokenizer saved!")
print(f"Location: {save_path}")

# List files
import os
saved_files = os.listdir(save_path)
print("\nFiles saved:")
for f in saved_files:
    print(f"  - {f}")


In [ ]:
from transformers import pipeline
import torch

# Reload from saved directory
chat_model = AutoModelForCausalLM.from_pretrained(save_path)
chat_tokenizer = AutoTokenizer.from_pretrained(save_path)
chat_tokenizer.pad_token = chat_tokenizer.eos_token

generator = pipeline(
    "text-generation",
    model=chat_model,
    tokenizer=chat_tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

def chat_with_mia(user_message, max_new_tokens=60, temperature=0.7, top_p=0.9):
    prompt = f"User: {user_message}\nMIA:"
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        num_return_sequences=1,
        pad_token_id=chat_tokenizer.eos_token_id,
    )
    full_text = outputs[0]["generated_text"]

    # Extract only MIA's part after "MIA:"
    if "MIA:" in full_text:
        mia_part = full_text.split("MIA:", 1)[1]
    else:
        mia_part = full_text

    # Stop if it starts a new "User:" segment
    mia_part = mia_part.split("User:", 1)[0]
    return mia_part.strip()

# Test with a few prompts
test_prompts = [
    "I'm feeling really lonely tonight",
    "I messed up at work today",
    "I'm anxious about my future",
]

print("=" * 60)
print("Testing MIA GPT-2 Model")
print("=" * 60)

for p in test_prompts:
    print(f"\nUser: {p}")
    response = chat_with_mia(p)
    print(f"MIA: {response}")
    print("-" * 60)
